<div style="background:linear-gradient(135deg,#117A65 0%,#1E8449 100%);padding:40px 36px 32px 36px;border-radius:10px;margin-bottom:8px;">
  <p style="color:#A9DFBF;font-size:13px;margin:0 0 6px 0;letter-spacing:2px;">CURSO 8 · MÓDULO 3 · CLASE 8</p>
  <h1 style="color:white;font-size:36px;margin:0 0 10px 0;font-weight:700;">Análisis Discriminante Lineal (LDA)</h1>
  <p style="color:#A9DFBF;font-size:16px;margin:0 0 24px 0;font-style:italic;">Fisher · Frontera de decisión · LDA vs Logística</p>
  <hr style="border-color:#52BE80;margin:0 0 20px 0;">
  <p style="color:#EAFAF1;font-size:13px;margin:0;">📌 <strong>Docente:</strong> Josef Rodriguez &nbsp;·&nbsp; <strong>Módulo 3:</strong> Análisis Discriminante &nbsp;·&nbsp; 2 horas</p>
</div>

## Objetivos
| # | Al terminar podés |
|---|-------------------|
| 1 | Derivar la regla de Fisher: maximizar separación entre clases |
| 2 | Calcular Sᴮ y Sᵥᵥ manualmente y encontrar la dirección de Fisher |
| 3 | Interpretar la frontera de decisión geométricamente |
| 4 | Comparar LDA con Logística en distintos escenarios |
| 5 | Aplicar LDA multiclase y visualizar la proyección discriminante |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({'figure.dpi':110,'font.size':11,'axes.spines.top':False,'axes.spines.right':False})
SEED=42; np.random.seed(SEED)
print('Setup listo')

---
## 1. La idea de Fisher

LDA maximiza:
$$J(\mathbf{w}) = \frac{\mathbf{w}^\top S_B \mathbf{w}}{\mathbf{w}^\top S_W \mathbf{w}}$$

Solución: **w** = Sᵥᵥ⁻¹(μ₁ − μ₀)

In [ ]:
np.random.seed(SEED)
n_c=150
X0=np.random.multivariate_normal([2,6],[[1.5,0.3],[0.3,1.2]],n_c)
X1=np.random.multivariate_normal([7,2],[[1.2,-0.3],[-0.3,1.5]],n_c)
X_all=np.vstack([X0,X1]); y_all=np.array([0]*n_c+[1]*n_c)

mu0,mu1=X0.mean(0),X1.mean(0)
Sw=(X0-mu0).T@(X0-mu0)+(X1-mu1).T@(X1-mu1)
Sb=n_c*np.outer(mu1-mu0,mu1-mu0)
w=np.linalg.inv(Sw)@(mu1-mu0); w/=np.linalg.norm(w)

z0=X0@w; z1=X1@w
th=(z0.mean()+z1.mean())/2

fig,axes=plt.subplots(1,2,figsize=(13,5))
axes[0].scatter(X0[:,0],X0[:,1],alpha=0.5,s=20,color='#E74C3C',label='Default (0)')
axes[0].scatter(X1[:,0],X1[:,1],alpha=0.5,s=20,color='#1E8449',label='No default (1)')
centro=X_all.mean(0)
axes[0].annotate('',xy=centro+3*w,xytext=centro-3*w,arrowprops=dict(arrowstyle='<->',color='#117A65',lw=2.5))
axes[0].set(xlabel='Ingresos',ylabel='Deuda',title='Dirección de Fisher'); axes[0].legend(); axes[0].grid(True,alpha=0.2)
axes[1].hist(z0,bins=28,alpha=0.6,color='#E74C3C',label='Default',density=True)
axes[1].hist(z1,bins=28,alpha=0.6,color='#1E8449',label='No default',density=True)
axes[1].axvline(th,color='#117A65',lw=2.5,linestyle='--',label=f'Frontera={th:.2f}')
axes[1].set(xlabel='z=wᵀx',title='Proyecciones'); axes[1].legend(); axes[1].grid(True,alpha=0.2)
plt.tight_layout(); plt.show()
print(f'w Fisher: {w.round(4)}')
print(f'Frontera: {th:.4f}')

In [ ]:
# Verificar con sklearn LDA
Xtr,Xte,ytr,yte=train_test_split(X_all,y_all,test_size=0.25,random_state=SEED)
lda_bin=LinearDiscriminantAnalysis()
lda_bin.fit(Xtr,ytr)
acc=accuracy_score(yte,lda_bin.predict(Xte))
print(f'LDA sklearn — Accuracy: {acc:.4f}')
w_skl=lda_bin.coef_[0]; w_skl/=np.linalg.norm(w_skl)
print(f'Similitud Fisher manual vs sklearn: {abs(w@w_skl):.6f} (debe ser 1)')

---
## 2. LDA multiclase — proyección en K−1 dimensiones

Con K clases, LDA reduce a K−1 dimensiones discriminantes.

In [ ]:
np.random.seed(SEED)
Sigma_c=np.array([[2.0,0.6,0.2],[0.6,2.0,0.4],[0.2,0.4,1.0]])
mus=[np.array([-2,-2,1]),np.array([1,2,-1]),np.array([3,-1,0])]
X_mc=np.vstack([np.random.multivariate_normal(mu,Sigma_c,150) for mu in mus])
y_mc=np.repeat([0,1,2],150)
Xtr_mc,Xte_mc,ytr_mc,yte_mc=train_test_split(X_mc,y_mc,test_size=0.25,random_state=SEED)
lda_mc=LinearDiscriminantAnalysis(); lda_mc.fit(Xtr_mc,ytr_mc)
acc_mc=accuracy_score(yte_mc,lda_mc.predict(Xte_mc))
print(f'LDA multiclase — Accuracy: {acc_mc:.4f}')
print(classification_report(yte_mc,lda_mc.predict(Xte_mc),target_names=['Clase A','Clase B','Clase C']))
X_proj=lda_mc.transform(Xte_mc)
fig,axes=plt.subplots(1,2,figsize=(13,5))
colores=['#E74C3C','#1E8449','#2E86C1']; labels_mc=['Clase A','Clase B','Clase C']
for cls,col,lbl in zip([0,1,2],colores,labels_mc):
    m=yte_mc==cls
    axes[0].scatter(Xte_mc[m,0],Xte_mc[m,1],s=20,alpha=0.6,color=col,label=lbl)
    axes[1].scatter(X_proj[m,0],X_proj[m,1],s=20,alpha=0.7,color=col,label=lbl)
axes[0].set(xlabel='F1',ylabel='F2',title='Espacio original'); axes[0].legend(); axes[0].grid(True,alpha=0.2)
axes[1].set(xlabel='LD1',ylabel='LD2',title=f'Espacio discriminante (Acc={acc_mc:.3f})')
axes[1].legend(); axes[1].grid(True,alpha=0.2)
plt.tight_layout(); plt.show()

---
## 3. LDA vs Logística — cuándo difieren

In [ ]:
np.random.seed(SEED)
n_cmp=400
X_normal=np.vstack([np.random.multivariate_normal([-1,-1],[[2,0.5],[0.5,1]],n_cmp//2),
                    np.random.multivariate_normal([1,1],[[2,0.5],[0.5,1]],n_cmp//2)])
y_normal=np.array([0]*(n_cmp//2)+[1]*(n_cmp//2))
X_outlier=X_normal.copy()
X_outlier[np.random.choice(n_cmp,30)]+=np.random.randn(30,2)*5
resultados=[]
for nombre,Xd in [('Normal (MVN OK)',X_normal),('Con outliers',X_outlier)]:
    Xtr_c,Xte_c,ytr_c,yte_c=train_test_split(Xd,y_normal,test_size=0.25,random_state=SEED)
    for nm_m,modelo in [('LDA',LinearDiscriminantAnalysis()),
                         ('Logistica',LogisticRegression(C=1e6,solver='lbfgs',max_iter=500))]:
        modelo.fit(Xtr_c,ytr_c)
        auc=roc_auc_score(yte_c,modelo.predict_proba(Xte_c)[:,1])
        acc=accuracy_score(yte_c,modelo.predict(Xte_c))
        resultados.append({'Dataset':nombre,'Modelo':nm_m,'AUC':round(auc,4),'Acc':round(acc,4)})
df_cmp=pd.DataFrame(resultados)
print(df_cmp.to_string(index=False))
print('\nCon MVN: LDA y Logistica similares. Con outliers: Logistica mas robusta.')

---
## 4. Aplicación: segmentación de riesgo bancario

In [ ]:
np.random.seed(SEED)
n_seg=600
Sigma_risk=np.array([[3,1,-0.5,0.5],[1,4,0.5,0],[-0.5,0.5,2,0.3],[0.5,0,0.3,1.5]])
mus_risk=[np.array([-2,-2,1,0.5]),np.array([0,0,-0.5,-0.5]),np.array([2,2,-1.5,-1])]
X_risk=np.vstack([np.random.multivariate_normal(mu,Sigma_risk,n_seg//3) for mu in mus_risk])
y_risk=np.repeat(['Alto','Medio','Bajo'],n_seg//3)
feats_risk=['deuda_ratio','historial_neg','ingresos','antiguedad']
sc_risk=StandardScaler(); X_risk_sc=sc_risk.fit_transform(X_risk)
Xtr_r,Xte_r,ytr_r,yte_r=train_test_split(X_risk_sc,y_risk,test_size=0.2,random_state=SEED)
lda_risk=LinearDiscriminantAnalysis(); lda_risk.fit(Xtr_r,ytr_r)
acc_r=accuracy_score(yte_r,lda_risk.predict(Xte_r))
print(f'Accuracy segmentacion de riesgo: {acc_r:.4f}')
print(classification_report(yte_r,lda_risk.predict(Xte_r)))
print('Coeficientes discriminantes:')
for nm,c1,c2 in zip(feats_risk,lda_risk.scalings_[:,0],lda_risk.scalings_[:,1]):
    print(f'  {nm:18s}: LD1={c1:+.4f}  LD2={c2:+.4f}')

In [ ]:
# Predecir nuevos clientes
nuevos_risk=np.array([[0.8,3,-0.5,0.3],[-0.5,-1,1.5,1.2],[0.2,0.5,0.5,0]])
nuevos_sc=sc_risk.transform(nuevos_risk)
preds=lda_risk.predict(nuevos_sc); probs=lda_risk.predict_proba(nuevos_sc)
classes=lda_risk.classes_
print('Prediccion de riesgo:')
for i,(pred,prob) in enumerate(zip(preds,probs)):
    p_dict=dict(zip(classes,prob.round(3)))
    print(f'  Cliente {chr(65+i)}: {pred:6s} | {p_dict}')

---
## Conclusiones

<div style="background:#EAFAF1;border-left:5px solid #1E8449;padding:20px 24px;border-radius:0 8px 8px 0;">

**01 · Fisher LDA = w = Sᵥᵥ⁻¹(μ₁ − μ₀)**
Dirección que maximiza la separación entre medias dividida por la dispersión interna.

**02 · LDA y Logística convergen cuando MVN se cumple**
Con outliers o distribuciones muy asimétricas, Logística es más robusta.

**03 · LDA es naturalmente multiclase — reduce a K−1 dimensiones**
Ideal para visualización y clasificación con muchos grupos.

</div>

---
<div style="background:#117A65;color:white;padding:20px 24px;border-radius:8px;">
<strong>Próxima clase — Clase 9</strong><br>
Scores discriminantes · QDA · Error por clase · Caso completo de marketing<br>
<em>Docente: Josef Rodriguez · Curso 8 · Módulo 3</em>
</div>